# Agent Loop & Tool Calling

## Overview

This document explains how the debug assistant's agent loop works — the core architectural pattern that makes it different from the single-turn RAG in the Document Q&A service. It covers how Ollama's tool-calling API works, how the observe-think-act loop is built from scratch, and the guardrails that keep a 14B model from going off the rails.

## Architecture Context

**Document Q&A** uses a single-turn pattern:
```
question → embed → search → one LLM call → answer
```
The LLM fires once and you're done. It can't fetch new information mid-response.

**Debug Assistant** uses a multi-turn agent loop:
```
description → LLM decides tool → execute → feed result back → LLM decides again → ... → diagnosis
```
The LLM drives the conversation. It calls tools, observes results, and keeps going until it has enough information to answer.

This **observe → think → act** cycle is the core pattern. Each iteration:
1. **Observe** — the model sees the full conversation history (messages list) including prior tool results
2. **Think** — the model decides what to do next (call a tool or give a final answer)
3. **Act** — we execute the tool (or emit the diagnosis) and append the result to history

The agent loop lives in `agent.py`. It calls Ollama's `/api/chat` endpoint with tool definitions included in the request payload. The model either calls a tool (returns `tool_calls`) or gives a final text answer (no `tool_calls`).

This is the same pattern used by OpenAI function calling, Claude tool use, LangChain AgentExecutor, and LangGraph — industry standard. We build it from scratch here to understand what those frameworks are doing under the hood.

## Package Introductions

No new packages. This notebook uses only `json`, `asyncio`, and `typing` from the standard library.

The two new *concepts* are the Ollama tool-calling protocol and why we chose Qwen 2.5 14B.

---

### Ollama `/api/chat` Tool Calling

Tool calling is not a separate package — it's a protocol. You send a list of tool definitions alongside your messages. The model reads those definitions and, when it needs external information, returns a structured response saying "call this function with these arguments."

**Defining a tool** — a JSON schema object in the request:
```python
{
    "type": "function",
    "function": {
        "name": "search_logs",
        "description": "Search application logs for a keyword",
        "parameters": {
            "type": "object",
            "properties": {
                "keyword": {"type": "string", "description": "The keyword to search for"}
            },
            "required": ["keyword"]
        }
    }
}
```

**When the model wants to call a tool**, the response looks like:
```json
{
    "message": {
        "role": "assistant",
        "content": "",
        "tool_calls": [
            {
                "function": {
                    "name": "search_logs",
                    "arguments": {"keyword": "ConnectionError"}
                }
            }
        ]
    }
}
```

**When the model is done**, the response has no `tool_calls`:
```json
{
    "message": {
        "role": "assistant",
        "content": "Based on the logs, the issue is a database connection timeout..."
    }
}
```

The absence of `tool_calls` is your signal to stop the loop and emit the final diagnosis.

---

### Why Qwen 2.5 14B

Mistral 7B (used in the Document Q&A service) can do basic tool calling, but it's less reliable for multi-step reasoning. At 7B parameters, it sometimes forgets it already tried a tool, calls tools in an illogical order, or fails to recognize when it has enough information to answer.

Qwen 2.5 14B was specifically fine-tuned for function calling. It produces cleaner tool call JSON, handles multi-step chains more reliably, and is better at knowing when to *stop* calling tools. On an RTX 3090 (24GB VRAM), 14B runs at ~30 tokens/sec — fast enough for an interactive tool.

Bigger isn't always better, but for an agent that drives its own tool-calling loop, reliability beats raw speed.

## Go/TS Comparison

| Concept | Go/TS | Python |
|---------|-------|--------|
| Agent loop | `for` loop with `switch` on action type | `for` loop with `if/elif` on `tool_calls` |
| Async generator (streaming results) | channels / async iterators | `async def` + `yield` |
| JSON schema for tool params | hand-written struct or generated | hand-written `dict` |
| Conversation history | append to `[]Message` slice | append to `messages` list |
| Duplicate detection | `map[string]bool` | `set` of strings |
| Forced stop condition | `ctx.Done()` / max iteration counter | `max_steps` integer counter |

The Go mental model for an agent loop looks like:
```go
for step := 0; step < maxSteps; step++ {
    result := callLLM(messages)
    switch result.ActionType {
    case "tool_call":
        output := executeTool(result.ToolName, result.Args)
        messages = append(messages, toolResultMsg(output))
    case "text":
        return result.Content // final answer
    }
}
```

Python is structurally identical — just different syntax. The async generator part (`yield`) replaces the Go pattern of sending events over a channel.

## Build It

All steps below use mock data. No Ollama needed — the notebook is fully runnable on Mac.

### Step 1: Ollama message format

The `/api/chat` endpoint takes a `messages` list. Each message has a `role` and `content`. Four roles matter here:
- `system` — the instructions that define the assistant's behavior
- `user` — the human's input
- `assistant` — the model's prior responses (including tool calls)
- `tool` — the result of a tool execution, fed back to the model

In [ ]:
import json

# A typical conversation history mid-agent-loop.
# The model sees ALL of this on every call.
messages = [
    {
        "role": "system",
        "content": (
            "You are a debug assistant. Use the available tools to investigate "
            "the issue, then provide a diagnosis."
        ),
    },
    {
        "role": "user",
        "content": "My app keeps crashing with a ConnectionError at midnight.",
    },
    {
        # The model decided to call search_logs
        "role": "assistant",
        "content": "",
        "tool_calls": [
            {
                "function": {
                    "name": "search_logs",
                    "arguments": {"keyword": "ConnectionError"},
                }
            }
        ],
    },
    {
        # We executed search_logs and fed the result back
        "role": "tool",
        "content": json.dumps(
            {
                "matches": [
                    "2024-01-15 00:00:03 ERROR ConnectionError: db timed out after 30s",
                    "2024-01-14 00:00:01 ERROR ConnectionError: db timed out after 30s",
                ]
            }
        ),
    },
]

print(f"Message history has {len(messages)} entries:")
for i, msg in enumerate(messages):
    role = msg["role"]
    if role == "assistant" and msg.get("tool_calls"):
        tc = msg["tool_calls"][0]["function"]
        preview = f"[tool_call: {tc['name']}({tc['arguments']})]"
    else:
        preview = str(msg.get("content", ""))[:60]
    print(f"  [{i}] {role}: {preview}")

The `tool` role is how results get back to the model. After we run `search_logs`, we append a `{"role": "tool", "content": "<result>"}` message. On the next loop iteration, the model reads this and decides what to do next.

---

### Step 2: Define a tool

Tools are JSON schema objects. The `description` field is critical — it's what the model reads to decide *when* to call the tool. A vague description leads to wrong tool calls.

In [ ]:
# Tool definitions sent to Ollama in the request payload.
# The model reads these to understand what tools are available.

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_logs",
            "description": (
                "Search application log files for a keyword. "
                "Returns matching log lines with timestamps. "
                "Use this to find error messages, stack traces, or specific events."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {
                        "type": "string",
                        "description": "The keyword or phrase to search for in logs",
                    }
                },
                "required": ["keyword"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_system_metrics",
            "description": (
                "Retrieve CPU, memory, and disk usage metrics for a time window. "
                "Use this to identify resource exhaustion or spikes correlated with errors."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "window_minutes": {
                        "type": "integer",
                        "description": "How many minutes back to retrieve metrics for",
                    }
                },
                "required": ["window_minutes"],
            },
        },
    },
]

print(f"Defined {len(TOOLS)} tools:")
for tool in TOOLS:
    fn = tool["function"]
    params = list(fn["parameters"]["properties"].keys())
    print(f"  {fn['name']}({', '.join(params)})")
    print(f"    → {fn['description'][:70]}...")

---

### Step 3: Simulate a tool call response

This is what Ollama actually returns when the model decides to call a tool. Let's parse it the same way the real agent loop does.

In [ ]:
# Simulated Ollama response — model wants to call search_logs
ollama_tool_response = {
    "model": "qwen2.5:14b",
    "message": {
        "role": "assistant",
        "content": "",
        "tool_calls": [
            {
                "function": {
                    "name": "search_logs",
                    "arguments": {"keyword": "ConnectionError"},
                }
            }
        ],
    },
    "done": True,
}

# Simulated Ollama response — model gives a final text answer
ollama_text_response = {
    "model": "qwen2.5:14b",
    "message": {
        "role": "assistant",
        "content": (
            "Based on the logs, the app crashes at midnight due to a database "
            "connection timeout. The DB likely has a scheduled maintenance window "
            "or connection pool reset at 00:00. Recommendation: increase the "
            "connection timeout or add retry logic around DB operations."
        ),
    },
    "done": True,
}


def parse_ollama_response(response: dict) -> tuple[bool, str | None, dict | None]:
    """Parse an Ollama /api/chat response.

    Returns (is_tool_call, tool_name, tool_args) or (False, None, None) for text.
    """
    message = response["message"]
    tool_calls = message.get("tool_calls")

    if tool_calls:
        # Take only the first tool call — models sometimes return multiple,
        # but we process one at a time to keep history clean.
        first = tool_calls[0]["function"]
        name = first["name"]
        arguments = first["arguments"]

        # Pitfall: Ollama sometimes returns arguments as a JSON *string*
        # instead of an already-parsed dict. Always normalize.
        if isinstance(arguments, str):
            arguments = json.loads(arguments)

        return True, name, arguments

    return False, None, None


# Parse the tool-call response
is_tool, name, args = parse_ollama_response(ollama_tool_response)
print(f"Tool call response → is_tool={is_tool}, name={name}, args={args}")

# Parse the text response
is_tool, name, args = parse_ollama_response(ollama_text_response)
text_content = ollama_text_response["message"]["content"]
print(f"Text response     → is_tool={is_tool}, content='{text_content[:50]}...'")  

> **Pitfall:** Ollama sometimes returns `arguments` as a JSON string (e.g., `'{"keyword": "error"}'`) instead of a parsed dict. This varies by model version. Always check `isinstance(arguments, str)` and call `json.loads()` if needed. If you don't, you'll get a `TypeError` when you try to pass the string as keyword arguments.

---

### Step 4: Build the minimal agent loop

Now we put it together. The loop:
1. Call the mock LLM
2. If it returns a tool call → execute the tool, append to history, continue
3. If it returns text → that's the final diagnosis, stop

We use a list of pre-scripted mock responses to simulate a two-step agent run.

In [ ]:
import asyncio

# ---------------------------------------------------------------------------
# Mock infrastructure — replaces real Ollama and real tools
# ---------------------------------------------------------------------------

# Pre-scripted sequence of Ollama responses.
# In the real service these come from httpx calls.
MOCK_LLM_RESPONSES = [
    # Step 1: model wants to search logs
    {
        "message": {
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {"function": {"name": "search_logs", "arguments": {"keyword": "ConnectionError"}}}
            ],
        }
    },
    # Step 2: model has enough info, gives final diagnosis
    {
        "message": {
            "role": "assistant",
            "content": (
                "The logs show repeated ConnectionError timeouts at midnight. "
                "This is consistent with a scheduled DB maintenance window. "
                "Add connection retry logic with exponential backoff."
            ),
        }
    },
]

response_index = 0


async def mock_ollama_call(messages: list, tools: list) -> dict:
    """Return the next pre-scripted response. Simulates /api/chat."""
    global response_index
    response = MOCK_LLM_RESPONSES[response_index % len(MOCK_LLM_RESPONSES)]
    response_index += 1
    return response


async def execute_tool(name: str, args: dict) -> str:
    """Mock tool executor. Returns fake results for any tool name."""
    if name == "search_logs":
        return json.dumps(
            {
                "matches": [
                    "2024-01-15 00:00:03 ERROR ConnectionError: db timed out",
                    "2024-01-14 00:00:01 ERROR ConnectionError: db timed out",
                ]
            }
        )
    if name == "get_system_metrics":
        return json.dumps({"cpu_pct": 12, "mem_pct": 45, "disk_pct": 60})
    return json.dumps({"error": f"unknown tool: {name}"})


# ---------------------------------------------------------------------------
# Minimal agent loop
# ---------------------------------------------------------------------------

async def run_agent(user_message: str, tools: list, max_steps: int = 10) -> str:
    """Run the observe-think-act loop until a diagnosis is produced.

    Returns the final diagnosis string.
    """
    messages = [
        {
            "role": "system",
            "content": "You are a debug assistant. Use tools to investigate, then diagnose.",
        },
        {"role": "user", "content": user_message},
    ]

    for step in range(max_steps):
        print(f"\n[step {step + 1}] Calling LLM with {len(messages)} messages...")

        # THINK: ask the model what to do next
        response = await mock_ollama_call(messages, tools)
        message = response["message"]
        tool_calls = message.get("tool_calls")

        if tool_calls:
            # ACT: model wants to call a tool
            first = tool_calls[0]["function"]
            name = first["name"]
            arguments = first["arguments"]
            if isinstance(arguments, str):
                arguments = json.loads(arguments)

            print(f"         → tool_call: {name}({arguments})")

            # Append the assistant's tool-call message to history
            messages.append(message)

            # OBSERVE: execute the tool and append the result
            result = await execute_tool(name, arguments)
            messages.append({"role": "tool", "content": result})
            print(f"         → tool_result: {result[:60]}...")

        else:
            # DONE: model produced a final text answer
            diagnosis = message["content"]
            print(f"         → diagnosis: '{diagnosis[:60]}...'")
            return diagnosis

    return "[max_steps reached — no diagnosis produced]"


# Run it
response_index = 0  # reset mock index
diagnosis = asyncio.run(run_agent(
    user_message="My app crashes with ConnectionError every night at midnight.",
    tools=TOOLS,
))

print(f"\n=== Final Diagnosis ===")
print(diagnosis)

That's the core loop. Under 40 lines including comments. The real `agent.py` is essentially this with real `httpx` calls instead of mocks.

---

### Step 5: Add guardrails

`max_steps` alone isn't enough. A model can hit `max_steps` by asking different questions, or it can spin in circles calling the same tool with the same arguments. Two guardrails handle this:

1. **Hard cap** — stop after `max_steps` regardless
2. **Duplicate detection** — track seen `(tool_name, args)` pairs; if the model repeats one, break early
3. **Forced diagnosis** — when stopping early, call Ollama *without tools* to get a summary of what was found so far

In [ ]:
# Guardrail 1: Duplicate detection
# We use a set of JSON strings as the key.
# Sorting keys ensures {"a":1,"b":2} and {"b":2,"a":1} match.

seen_calls: set[str] = set()

def is_duplicate(tool_name: str, arguments: dict) -> bool:
    key = json.dumps({"name": tool_name, "args": arguments}, sort_keys=True)
    if key in seen_calls:
        return True
    seen_calls.add(key)
    return False


# Demo: first call is new, second identical call is a duplicate
seen_calls.clear()
print(is_duplicate("search_logs", {"keyword": "error"}))   # False — first time
print(is_duplicate("search_logs", {"keyword": "error"}))   # True  — duplicate
print(is_duplicate("search_logs", {"keyword": "warning"})) # False — different args
print(is_duplicate("get_system_metrics", {"window_minutes": 5})) # False — different tool

In [ ]:
# Guardrail 2: Forced diagnosis
# When max_steps is reached or a duplicate is detected, we call Ollama
# WITHOUT tools in the payload. This forces a text response.

# Simulated forced-diagnosis response (what the model returns with no tools)
FORCED_DIAGNOSIS_RESPONSE = {
    "message": {
        "role": "assistant",
        "content": (
            "Based on the investigation so far: logs show repeated ConnectionError "
            "at midnight. Full root cause analysis was incomplete, but the pattern "
            "suggests a scheduled DB reset. Recommend adding retry logic."
        ),
    }
}


async def force_diagnosis(messages: list) -> str:
    """Call the LLM without tools to force a text diagnosis.

    Passing an empty tools list (or omitting it) prevents the model
    from trying to call another tool instead of answering.
    """
    # In real code: await ollama_client.chat(messages=messages, tools=[])
    # Mock version:
    print("  [force_diagnosis] Calling LLM with tools=[] to force text response")
    return FORCED_DIAGNOSIS_RESPONSE["message"]["content"]


# Show the full guarded loop
async def run_agent_with_guardrails(
    user_message: str,
    tools: list,
    max_steps: int = 10,
) -> str:
    messages = [
        {"role": "system", "content": "You are a debug assistant. Use tools to investigate, then diagnose."},
        {"role": "user", "content": user_message},
    ]
    seen_calls: set[str] = set()

    for step in range(max_steps):
        print(f"\n[step {step + 1}/{max_steps}]")
        response = await mock_ollama_call(messages, tools)
        message = response["message"]
        tool_calls = message.get("tool_calls")

        if not tool_calls:
            # Natural completion — model gave a text answer
            print("  → natural completion")
            return message["content"]

        first = tool_calls[0]["function"]
        name = first["name"]
        arguments = first["arguments"]
        if isinstance(arguments, str):
            arguments = json.loads(arguments)

        # Guardrail: duplicate detection
        key = json.dumps({"name": name, "args": arguments}, sort_keys=True)
        if key in seen_calls:
            print(f"  → duplicate call detected: {name}({arguments}) — forcing diagnosis")
            return await force_diagnosis(messages)
        seen_calls.add(key)

        print(f"  → tool_call: {name}({arguments})")
        messages.append(message)
        result = await execute_tool(name, arguments)
        messages.append({"role": "tool", "content": result})
        print(f"  → tool_result appended ({len(result)} chars)")

    # Guardrail: max_steps reached
    print(f"\n  → max_steps ({max_steps}) reached — forcing diagnosis")
    return await force_diagnosis(messages)


response_index = 0
result = asyncio.run(run_agent_with_guardrails(
    user_message="App crashes with ConnectionError at midnight.",
    tools=TOOLS,
    max_steps=10,
))
print(f"\n=== Diagnosis ===\n{result}")

> **Pitfall:** When forcing a final diagnosis at `max_steps`, call Ollama **without tools** in the payload. If you include tools, the model may try to call another tool instead of giving a text answer — sending you back into an infinite loop. An empty `tools=[]` signals "answer now."

---

### Step 6: Make it stream

The agent loop above blocks until it returns a final string. For a debug assistant, users want to see what's happening — which tools are being called, what results came back, and the diagnosis as it forms.

We convert the loop into an `AsyncGenerator` that yields typed event dicts as things happen.

In [ ]:
from typing import AsyncGenerator

# Event types the agent yields:
#
#   {"type": "thinking",    "step": 1}
#   {"type": "tool_call",   "name": "search_logs", "args": {...}}
#   {"type": "tool_result", "name": "search_logs", "result": "...", "step": 1}
#   {"type": "diagnosis",   "content": "..."}
#   {"type": "done"}


async def stream_agent(
    user_message: str,
    tools: list,
    max_steps: int = 10,
) -> AsyncGenerator[dict, None]:
    """Run the agent loop, yielding events as they happen.

    Consumers (e.g. an SSE endpoint) iterate this generator and
    forward each event to the client in real time.
    """
    messages = [
        {"role": "system", "content": "You are a debug assistant. Use tools to investigate, then diagnose."},
        {"role": "user", "content": user_message},
    ]
    seen_calls: set[str] = set()

    for step in range(max_steps):
        yield {"type": "thinking", "step": step + 1}

        response = await mock_ollama_call(messages, tools)
        message = response["message"]
        tool_calls = message.get("tool_calls")

        if not tool_calls:
            # Natural completion
            yield {"type": "diagnosis", "content": message["content"]}
            yield {"type": "done"}
            return

        first = tool_calls[0]["function"]
        name = first["name"]
        arguments = first["arguments"]
        if isinstance(arguments, str):
            arguments = json.loads(arguments)

        # Duplicate guard
        key = json.dumps({"name": name, "args": arguments}, sort_keys=True)
        if key in seen_calls:
            diagnosis = await force_diagnosis(messages)
            yield {"type": "diagnosis", "content": diagnosis}
            yield {"type": "done"}
            return
        seen_calls.add(key)

        yield {"type": "tool_call", "name": name, "args": arguments}
        messages.append(message)

        result = await execute_tool(name, arguments)
        messages.append({"role": "tool", "content": result})
        yield {"type": "tool_result", "name": name, "result": result, "step": step + 1}

    # max_steps reached
    diagnosis = await force_diagnosis(messages)
    yield {"type": "diagnosis", "content": diagnosis}
    yield {"type": "done"}


# Consume the stream
async def demo_stream():
    global response_index
    response_index = 0

    print("Streaming agent events:\n")
    async for event in stream_agent(
        user_message="App crashes with ConnectionError at midnight.",
        tools=TOOLS,
    ):
        event_type = event["type"]
        if event_type == "thinking":
            print(f"[thinking] step {event['step']}")
        elif event_type == "tool_call":
            print(f"[tool_call] {event['name']}({event['args']})")
        elif event_type == "tool_result":
            print(f"[tool_result] {event['name']} → {event['result'][:50]}...")
        elif event_type == "diagnosis":
            print(f"[diagnosis] {event['content'][:80]}...")
        elif event_type == "done":
            print("[done]")


asyncio.run(demo_stream())

The SSE endpoint in `main.py` wraps this generator exactly the same way the Document Q&A service wraps `stream_ollama_response()` — iterating the generator and sending each event as `data: {json}\n\n`.

The frontend reads these events and updates the UI: "Thinking... → Searching logs... → Found 2 matches → Diagnosis: ..."

## Experiment

### Experiment 1: Does the model call tools if the system prompt doesn't mention them?

Tools are defined in the request payload — separate from the system prompt. The model can see them regardless of what the prompt says.

In [ ]:
# In the real agent, tools are passed in the request body:
#
#   POST /api/chat
#   {
#       "model": "qwen2.5:14b",
#       "messages": [...],
#       "tools": [...]    ← tools are here, not in the system prompt
#   }
#
# The system prompt affects *how* the model behaves, but the model
# always has access to whatever tools are in the payload.

# System prompt WITHOUT mentioning tools
minimal_prompt_messages = [
    {"role": "system", "content": "You are a helpful assistant."},  # no mention of tools
    {"role": "user", "content": "What's wrong with my app? It crashes every night."},
]

# Simulated response: model still calls a tool because tools are in the payload
# (In a real run, Qwen 2.5 14B would still use the tools — they're always visible)
print("Tools are defined in the request payload, not the system prompt.")
print("The model can call them regardless of whether the prompt mentions them.")
print()
print("However — a good system prompt HELPS by telling the model:")
print("  1. When to stop calling tools (avoid over-investigation)")
print("  2. What format to use for the final diagnosis")
print("  3. What tools are available and what each one is good for")
print()
print("Tools in prompt vs tools in payload:")
print("  payload tools  → the model CAN call these (capability)")
print("  prompt mention → guides WHEN and HOW to call them (behavior)")

### Experiment 2: What happens when `max_steps=1`?

Setting `max_steps=1` means: one chance to call a tool, then immediately force a diagnosis. Walk through the event sequence.

In [ ]:
# With max_steps=1:
#   Step 1: model returns tool_call → we execute it → we hit max_steps
#   force_diagnosis() is called with the tool result in history
#   A diagnosis is produced from a single tool call

async def demo_max_steps_1():
    global response_index
    response_index = 0

    print("=== max_steps=1 event sequence ===")
    events = []
    async for event in stream_agent(
        user_message="App crashes at midnight.",
        tools=TOOLS,
        max_steps=1,  # only one step allowed
    ):
        events.append(event)
        print(f"  event: {event}")

    print(f"\nTotal events: {len(events)}")
    event_types = [e["type"] for e in events]
    print(f"Event types:  {event_types}")
    print()
    print("Expected: thinking → tool_call → tool_result → (max_steps hit) → diagnosis → done")
    print("Notice: force_diagnosis() is called after step 1 exhausts max_steps.")
    print("The tool result IS in the messages list, so the forced diagnosis can reference it.")


asyncio.run(demo_max_steps_1())

This is useful for testing: `max_steps=1` gives you the simplest possible agent run where you can verify the forced-diagnosis path works correctly before testing longer chains.

## Check Your Understanding

1. **Why did we build a custom agent loop instead of using LangChain's AgentExecutor or LangGraph?**

   Both frameworks are excellent, but they abstract away the mechanics we need to understand for a portfolio project. Building from scratch means we know exactly what's happening at each step — which tool call was made, what the raw response looked like, where the guardrails fire. LangChain/LangGraph also add significant dependency weight and their own opinions about state management. For a focused debug assistant with a well-defined tool set, the custom loop is ~50 lines and fully understandable.

2. **The duplicate detection uses a JSON string of `{name, args}` as the key. What edge case could this miss?**

   Argument ordering. `{"a": 1, "b": 2}` and `{"b": 2, "a": 1}` are semantically identical but produce different JSON strings without `sort_keys=True`. The implementation uses `json.dumps(..., sort_keys=True)` to normalize key order, but a subtler issue remains: floating point values that are numerically equal can serialize differently (`1.0` vs `1`). For the debug assistant's tools (which take string and integer arguments), this isn't a concern in practice.

3. **Why do we process only the first tool call even if the model returns multiple `tool_calls`?**

   Processing multiple tool calls in parallel would mean executing tools concurrently and appending multiple results before the next LLM call. While possible, it complicates history management: the model expects a `tool` message corresponding to each `tool_call` in order, and the result ordering must match. Processing one at a time keeps the conversation history simple and sequential — the model always sees one tool → one result before deciding the next action. For a debug assistant where tool order matters (you want to read logs before checking metrics), sequential processing is also semantically correct.